In [9]:
!unzip tomatoes_dataset.zip -d /content/datasets/tomatoes

Archive:  tomatoes_dataset.zip
   creating: /content/datasets/tomatoes/tomatoes_dataset/11-14 days/
  inflating: /content/datasets/tomatoes/tomatoes_dataset/11-14 days/eleven days (1).jpg  
  inflating: /content/datasets/tomatoes/tomatoes_dataset/11-14 days/eleven days (10).jpg  
  inflating: /content/datasets/tomatoes/tomatoes_dataset/11-14 days/eleven days (100).jpg  
  inflating: /content/datasets/tomatoes/tomatoes_dataset/11-14 days/eleven days (11).jpg  
  inflating: /content/datasets/tomatoes/tomatoes_dataset/11-14 days/eleven days (12).jpg  
  inflating: /content/datasets/tomatoes/tomatoes_dataset/11-14 days/eleven days (13).jpg  
  inflating: /content/datasets/tomatoes/tomatoes_dataset/11-14 days/eleven days (14).jpg  
  inflating: /content/datasets/tomatoes/tomatoes_dataset/11-14 days/eleven days (15).jpg  
  inflating: /content/datasets/tomatoes/tomatoes_dataset/11-14 days/eleven days (16).jpg  
  inflating: /content/datasets/tomatoes/tomatoes_dataset/11-14 days/eleven days (

In [10]:
import os
import shutil

base_dir = '/content/datasets'
combined_dir = '/content/combined_dataset'
os.makedirs(combined_dir, exist_ok=True)

foods = ['tomatoes']

food_stages = {

    'tomatoes': ['1-2 days', '3-5 days', '6-7 days', '8-11 days', '11-14 days']
}

for food in foods:
    food_path = os.path.join(base_dir, food)
    if os.path.exists(food_path):
        nested_folder = None
        for item in os.listdir(food_path):
            full_path = os.path.join(food_path, item)
            if os.path.isdir(full_path) and item.lower() == f"{food}_dataset".lower():
                nested_folder = full_path
                break
        if nested_folder:
            actual_stages = [d for d in os.listdir(nested_folder) if os.path.isdir(os.path.join(nested_folder, d))]
        else:
            actual_stages = [d for d in os.listdir(food_path) if os.path.isdir(os.path.join(food_path, d))]
        print(f"Checking {food}: Nested folder? {nested_folder}, Expected stages: {food_stages[food]}, Actual stages: {actual_stages}")
        for stage in food_stages[food]:
            src = os.path.join(nested_folder if nested_folder else food_path, stage)
            if os.path.exists(src):
                dst_class = f"{food}_{stage}"
                dst = os.path.join(combined_dir, dst_class)
                os.makedirs(dst, exist_ok=True)
                for img in os.listdir(src):
                    if os.path.isfile(os.path.join(src, img)):
                        shutil.copy(os.path.join(src, img), os.path.join(dst, img))
            else:
                print(f"Warning: {src} not found for {food}")

print("Combined dataset classes:", os.listdir(combined_dir))

Checking tomatoes: Nested folder? /content/datasets/tomatoes/tomatoes_dataset, Expected stages: ['1-2 days', '3-5 days', '6-7 days', '8-11 days', '11-14 days'], Actual stages: ['11-14 days', '8-11 days', '1-2 days', '6-7 days', '3-5 days']
Combined dataset classes: ['tomatoes_1-2 days', 'tomatoes_6-7 days', 'tomatoes_11-14 days', 'tomatoes_8-11 days', 'tomatoes_3-5 days']


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3, MobileNetV3Large
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import class_weight
import random
import os

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

# Custom augmentation function
def custom_augmentation(image):
    """Apply advanced augmentation techniques"""
    if random.random() < 0.3:  # Cutout with 30% probability
        h, w = image.shape[:2]
        cutout_size = random.randint(15, 50)
        x = random.randint(0, w - cutout_size)
        y = random.randint(0, h - cutout_size)
        image[y:y+cutout_size, x:x+cutout_size, :] = 0
    return image

def preprocess_with_augmentation(x):
    """Combined preprocessing and augmentation"""
    x = tf.keras.applications.efficientnet.preprocess_input(x)
    x = custom_augmentation(x)
    return x

# Enhanced data generators with stronger augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_with_augmentation,
    validation_split=0.15,  # Smaller validation split for more training data
    rotation_range=45,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=[0.7, 1.3],
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.6, 1.4],
    channel_shift_range=30.0,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=0.15
)

# Load data (adjust path as needed)
data_dir = '/content/combined_dataset'

train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(300, 300),  # Larger input size for better feature extraction
    batch_size=16,  # Smaller batch size for better gradient updates
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    data_dir,
    target_size=(300, 300),
    batch_size=16,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print(f"Classes: {train_generator.class_indices}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")

# Compute class weights for balanced training
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights_dict = dict(enumerate(class_weights))

# Build improved model with EfficientNetB3
def create_improved_model(num_classes=5):
    # Use EfficientNetB3 for better performance
    base_model = EfficientNetB3(
        weights='imagenet',
        include_top=False,
        input_shape=(300, 300, 3),
        drop_connect_rate=0.3  # Built-in regularization
    )

    # Add custom classification head
    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    # First dense layer with batch normalization
    x = Dense(1024, activation='relu', kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Second dense layer
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    # Output layer
    predictions = Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    return model, base_model

# Create the model
model, base_model = create_improved_model(5)

# Initial training with frozen base layers
for layer in base_model.layers:
    layer.trainable = False

# Compile with lower learning rate
model.compile(
    optimizer=AdamW(learning_rate=0.001, weight_decay=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy', 'top_1_accuracy']
)

print("Model Summary:")
print(f"Total parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]):,}")

# Enhanced callbacks
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.3,
        patience=5,
        min_lr=1e-8,
        mode='max',
        verbose=1
    ),
    ModelCheckpoint(
        'best_tomato_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

# Phase 1: Train classifier head
print("\n" + "="*50)
print("PHASE 1: Training classifier head only")
print("="*50)

history1 = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=30,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks,
    class_weight=class_weights_dict,
    verbose=1
)

# Phase 2: Fine-tune with unfrozen layers
print("\n" + "="*50)
print("PHASE 2: Fine-tuning with unfrozen layers")
print("="*50)

# Unfreeze the top layers of the base model
for layer in base_model.layers[-100:]:  # Unfreeze top 100 layers
    layer.trainable = True

# Recompile with very low learning rate
model.compile(
    optimizer=AdamW(learning_rate=0.0001, weight_decay=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy', 'top_1_accuracy']
)

# Reset early stopping patience for fine-tuning
callbacks[0].patience = 20

history2 = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=40,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks,
    class_weight=class_weights_dict,
    verbose=1
)

# Phase 3: Final fine-tuning with minimal learning rate
print("\n" + "="*50)
print("PHASE 3: Final fine-tuning")
print("="*50)

# Unfreeze all layers
for layer in base_model.layers:
    layer.trainable = True

# Very low learning rate for final tuning
model.compile(
    optimizer=AdamW(learning_rate=0.00001, weight_decay=0.000001),
    loss='categorical_crossentropy',
    metrics=['accuracy', 'top_1_accuracy']
)

callbacks[0].patience = 25

history3 = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=30,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks,
    class_weight=class_weights_dict,
    verbose=1
)

# Final evaluation
print("\n" + "="*50)
print("FINAL EVALUATION")
print("="*50)

# Load best model
model = tf.keras.models.load_model('best_tomato_model.keras')

# Evaluate on validation set
val_loss, val_acc, val_top1 = model.evaluate(val_generator, verbose=1)
print(f"Final Validation Accuracy: {val_acc * 100:.2f}%")

# Test Time Augmentation for even better results
def test_time_augmentation(model, generator, num_augs=10):
    """Apply test time augmentation for improved accuracy"""
    tta_datagen = ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        validation_split=0.15
    )

    predictions = []
    for i in range(num_augs):
        tta_gen = tta_datagen.flow_from_directory(
            data_dir,
            target_size=(300, 300),
            batch_size=16,
            class_mode='categorical',
            subset='validation',
            shuffle=False,
            seed=42 + i
        )
        pred = model.predict(tta_gen, verbose=0)
        predictions.append(pred)

    # Average predictions
    avg_predictions = np.mean(predictions, axis=0)
    y_true = val_generator.classes[:len(avg_predictions)]
    y_pred = np.argmax(avg_predictions, axis=1)

    accuracy = np.mean(y_pred == y_true)
    return accuracy * 100

print("\nApplying Test Time Augmentation...")
tta_accuracy = test_time_augmentation(model, val_generator)
print(f"TTA Validation Accuracy: {tta_accuracy:.2f}%")

# Plot training history
def plot_training_history(histories, title="Training History"):
    """Plot combined training history from multiple phases"""
    train_acc = []
    val_acc = []
    train_loss = []
    val_loss = []

    for hist in histories:
        train_acc.extend(hist.history.get('accuracy', []))
        val_acc.extend(hist.history.get('val_accuracy', []))
        train_loss.extend(hist.history.get('loss', []))
        val_loss.extend(hist.history.get('val_loss', []))

    epochs = range(1, len(train_acc) + 1)

    plt.figure(figsize=(15, 5))

    # Accuracy plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, [acc * 100 for acc in train_acc], 'bo-', label='Training Accuracy', alpha=0.8)
    plt.plot(epochs, [acc * 100 for acc in val_acc], 'ro-', label='Validation Accuracy', alpha=0.8)
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_loss, 'bo-', label='Training Loss', alpha=0.8)
    plt.plot(epochs, val_loss, 'ro-', label='Validation Loss', alpha=0.8)
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Plot the combined training history
plot_training_history([history1, history2, history3])

# Save final model
model.save('final_tomato_model_90plus.keras')
print(f"\nModel saved as 'final_tomato_model_90plus.keras'")

# Create a prediction function
def predict_tomato_stage(image_path):
    """Predict tomato stage from image path"""
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(300, 300))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)
    img_array = tf.keras.applications.efficientnet.preprocess_input(img_array)

    predictions = model.predict(img_array, verbose=0)
    predicted_class = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class]

    class_names = list(train_generator.class_indices.keys())
    return class_names[predicted_class], confidence

print("\nModel training completed!")
print("Expected accuracy: >90%")
print("Use predict_tomato_stage(image_path) to make predictions on new images")

Found 850 images belonging to 5 classes.
Found 150 images belonging to 5 classes.
Classes: {'tomatoes_1-2 days': 0, 'tomatoes_11-14 days': 1, 'tomatoes_3-5 days': 2, 'tomatoes_6-7 days': 3, 'tomatoes_8-11 days': 4}
Training samples: 850
Validation samples: 150


TypeError: EfficientNetB3() got an unexpected keyword argument 'drop_connect_rate'

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3, MobileNetV3Large
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import class_weight
import random
import os

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

# Custom augmentation function
def custom_augmentation(image):
    """Apply advanced augmentation techniques"""
    if random.random() < 0.3:  # Cutout with 30% probability
        h, w = image.shape[:2]
        cutout_size = random.randint(15, 50)
        x = random.randint(0, w - cutout_size)
        y = random.randint(0, h - cutout_size)
        image[y:y+cutout_size, x:x+cutout_size, :] = 0
    return image

def preprocess_with_augmentation(x):
    """Combined preprocessing and augmentation"""
    x = tf.keras.applications.efficientnet.preprocess_input(x)
    x = custom_augmentation(x)
    return x

# Enhanced data generators with stronger augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_with_augmentation,
    validation_split=0.15,  # Smaller validation split for more training data
    rotation_range=45,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=[0.7, 1.3],
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.6, 1.4],
    channel_shift_range=30.0,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=0.15
)

# Load data (adjust path as needed)
data_dir = '/content/combined_dataset'

train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(300, 300),  # Larger input size for better feature extraction
    batch_size=16,  # Smaller batch size for better gradient updates
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    data_dir,
    target_size=(300, 300),
    batch_size=16,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print(f"Classes: {train_generator.class_indices}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")

# Compute class weights for balanced training
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights_dict = dict(enumerate(class_weights))

# Build improved model with EfficientNetB3
def create_improved_model(num_classes=5):
    # Use EfficientNetB3 for better performance
    base_model = EfficientNetB3(
        weights='imagenet',
        include_top=False,
        input_shape=(300, 300, 3)
    )

    # Add custom classification head
    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    # First dense layer with batch normalization
    x = Dense(1024, activation='relu', kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Second dense layer
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    # Output layer
    predictions = Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    return model, base_model

# Create the model
model, base_model = create_improved_model(5)

# Initial training with frozen base layers
for layer in base_model.layers:
    layer.trainable = False

# Compile with lower learning rate
model.compile(
    optimizer=AdamW(learning_rate=0.001, weight_decay=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']  # Removed 'top_1_accuracy'
)

print("Model Summary:")
print(f"Total parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]):,}")

# Enhanced callbacks
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.3,
        patience=5,
        min_lr=1e-8,
        mode='max',
        verbose=1
    ),
    ModelCheckpoint(
        'best_tomato_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

# Phase 1: Train classifier head
print("\n" + "="*50)
print("PHASE 1: Training classifier head only")
print("="*50)

history1 = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=30,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks,
    class_weight=class_weights_dict,
    verbose=1
)

# Phase 2: Fine-tune with unfrozen layers
print("\n" + "="*50)
print("PHASE 2: Fine-tuning with unfrozen layers")
print("="*50)

# Unfreeze the top layers of the base model
for layer in base_model.layers[-100:]:  # Unfreeze top 100 layers
    layer.trainable = True

# Recompile with very low learning rate
model.compile(
    optimizer=AdamW(learning_rate=0.0001, weight_decay=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']  # Removed 'top_1_accuracy'
)

# Reset early stopping patience for fine-tuning
callbacks[0].patience = 20

history2 = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=40,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks,
    class_weight=class_weights_dict,
    verbose=1
)

# Phase 3: Final fine-tuning with minimal learning rate
print("\n" + "="*50)
print("PHASE 3: Final fine-tuning")
print("="*50)

# Unfreeze all layers
for layer in base_model.layers:
    layer.trainable = True

# Very low learning rate for final tuning
model.compile(
    optimizer=AdamW(learning_rate=0.00001, weight_decay=0.000001),
    loss='categorical_crossentropy',
    metrics=['accuracy']  # Removed 'top_1_accuracy'
)

callbacks[0].patience = 25

history3 = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=30,
    validation_data=val_generator,
    validation_steps=len(val_generator),
    callbacks=callbacks,
    class_weight=class_weights_dict,
    verbose=1
)

# Final evaluation
print("\n" + "="*50)
print("FINAL EVALUATION")
print("="*50)

# Load best model
model = tf.keras.models.load_model('best_tomato_model.keras')

# Evaluate on validation set
val_loss, val_acc = model.evaluate(val_generator, verbose=1)
print(f"Final Validation Accuracy: {val_acc * 100:.2f}%")

# Test Time Augmentation for even better results
def test_time_augmentation(model, generator, num_augs=10):
    """Apply test time augmentation for improved accuracy"""
    tta_datagen = ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        validation_split=0.15
    )

    predictions = []
    for i in range(num_augs):
        tta_gen = tta_datagen.flow_from_directory(
            data_dir,
            target_size=(300, 300),
            batch_size=16,
            class_mode='categorical',
            subset='validation',
            shuffle=False,
            seed=42 + i
        )
        pred = model.predict(tta_gen, verbose=0)
        predictions.append(pred)

    # Average predictions
    avg_predictions = np.mean(predictions, axis=0)
    y_true = val_generator.classes[:len(avg_predictions)]
    y_pred = np.argmax(avg_predictions, axis=1)

    accuracy = np.mean(y_pred == y_true)
    return accuracy * 100

print("\nApplying Test Time Augmentation...")
tta_accuracy = test_time_augmentation(model, val_generator)
print(f"TTA Validation Accuracy: {tta_accuracy:.2f}%")

# Plot training history
def plot_training_history(histories, title="Training History"):
    """Plot combined training history from multiple phases"""
    train_acc = []
    val_acc = []
    train_loss = []
    val_loss = []

    for hist in histories:
        train_acc.extend(hist.history.get('accuracy', []))
        val_acc.extend(hist.history.get('val_accuracy', []))
        train_loss.extend(hist.history.get('loss', []))
        val_loss.extend(hist.history.get('val_loss', []))

    epochs = range(1, len(train_acc) + 1)

    plt.figure(figsize=(15, 5))

    # Accuracy plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, [acc * 100 for acc in train_acc], 'bo-', label='Training Accuracy', alpha=0.8)
    plt.plot(epochs, [acc * 100 for acc in val_acc], 'ro-', label='Validation Accuracy', alpha=0.8)
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_loss, 'bo-', label='Training Loss', alpha=0.8)
    plt.plot(epochs, val_loss, 'ro-', label='Validation Loss', alpha=0.8)
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Plot the combined training history
plot_training_history([history1, history2, history3])

# Save final model
model.save('final_tomato_model_90plus.keras')
print(f"\nModel saved as 'final_tomato_model_90plus.keras'")

# Create a prediction function
def predict_tomato_stage(image_path):
    """Predict tomato stage from image path"""
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(300, 300))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)
    img_array = tf.keras.applications.efficientnet.preprocess_input(img_array)

    predictions = model.predict(img_array, verbose=0)
    predicted_class = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class]

    class_names = list(train_generator.class_indices.keys())
    return class_names[predicted_class], confidence

print("\nModel training completed!")
print("Expected accuracy: >90%")
print("Use predict_tomato_stage(image_path) to make predictions on new images")

Found 850 images belonging to 5 classes.
Found 150 images belonging to 5 classes.
Classes: {'tomatoes_1-2 days': 0, 'tomatoes_11-14 days': 1, 'tomatoes_3-5 days': 2, 'tomatoes_6-7 days': 3, 'tomatoes_8-11 days': 4}
Training samples: 850
Validation samples: 150
Model Summary:
Total parameters: 12,890,932
Trainable parameters: 2,104,325

PHASE 1: Training classifier head only


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.6114 - loss: 1.3928
Epoch 1: val_accuracy improved from -inf to 0.77333, saving model to best_tomato_model.keras
54/54 ━━━━━━━━━━━━━━━━━━━━ 354s 6s/step - accuracy: 0.6137 - loss: 1.3865 - val_accuracy: 0.7733 - val_loss: 0.7622 - learning_rate: 0.0010
Epoch 2/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.8540 - loss: 0.6487
Epoch 2: val_accuracy improved from 0.77333 to 0.85333, saving model to best_tomato_model.keras
54/54 ━━━━━━━━━━━━━━━━━━━━ 320s 6s/step - accuracy: 0.8540 - loss: 0.6488 - val_accuracy: 0.8533 - val_loss: 0.7071 - learning_rate: 0.0010
Epoch 3/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.8846 - loss: 0.5522
Epoch 3: val_accuracy did not improve from 0.85333
54/54 ━━━━━━━━━━━━━━━━━━━━ 341s 6s/step - accuracy: 0.8844 - loss: 0.5535 - val_accuracy: 0.7867 - val_loss: 0.8417 - learning_rate: 0.0010
Epoch 4/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.8443 - loss: 0.6674
Epoch 4:

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt

# Define data directory and validation generator
data_dir = '/content/combined_dataset'
val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    validation_split=0.15
)
val_generator = val_datagen.flow_from_directory(
    data_dir,
    target_size=(300, 300),
    batch_size=16,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Load best model
model = tf.keras.models.load_model('/content/best_tomato_model .keras')

# Evaluate on validation set
val_loss, val_acc = model.evaluate(val_generator, verbose=1)
print(f"Final Validation Accuracy: {val_acc * 100:.2f}%")

# Test Time Augmentation for even better results
def test_time_augmentation(model, generator, num_augs=10):
    """Apply test time augmentation for improved accuracy"""
    tta_datagen = ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        validation_split=0.15
    )

    predictions = []
    for i in range(num_augs):
        tta_gen = tta_datagen.flow_from_directory(
            data_dir,
            target_size=(300, 300),
            batch_size=16,
            class_mode='categorical',
            subset='validation',
            shuffle=False,
            seed=42 + i
        )
        pred = model.predict(tta_gen, verbose=0)
        predictions.append(pred)

    # Average predictions
    avg_predictions = np.mean(predictions, axis=0)
    y_true = val_generator.classes[:len(avg_predictions)]
    y_pred = np.argmax(avg_predictions, axis=1)

    accuracy = np.mean(y_pred == y_true)
    return accuracy * 100

print("\nApplying Test Time Augmentation...")
tta_accuracy = test_time_augmentation(model, val_generator)
print(f"TTA Validation Accuracy: {tta_accuracy:.2f}%")

# Plot training history
def plot_training_history(histories, title="Training History"):
    """Plot combined training history from multiple phases"""
    train_acc = []
    val_acc = []
    train_loss = []
    val_loss = []

    for hist in histories:
        train_acc.extend(hist.history.get('accuracy', []))
        val_acc.extend(hist.history.get('val_accuracy', []))
        train_loss.extend(hist.history.get('loss', []))
        val_loss.extend(hist.history.get('val_loss', []))

    epochs = range(1, len(train_acc) + 1)

    plt.figure(figsize=(15, 5))

    # Accuracy plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, [acc * 100 for acc in train_acc], 'bo-', label='Training Accuracy', alpha=0.8)
    plt.plot(epochs, [acc * 100 for acc in val_acc], 'ro-', label='Validation Accuracy', alpha=0.8)
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_loss, 'bo-', label='Training Loss', alpha=0.8)
    plt.plot(epochs, val_loss, 'ro-', label='Validation Loss', alpha=0.8)
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()



# Save final model
model.save('final_tomato_model_90plus.keras')
print(f"\nModel saved as 'final_tomato_model_90plus.keras'")

# Create a prediction function
def predict_tomato_stage(image_path):
    """Predict tomato stage from image path"""
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(300, 300))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)
    img_array = tf.keras.applications.efficientnet.preprocess_input(img_array)

    predictions = model.predict(img_array, verbose=0)
    predicted_class = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class]

    class_names = list(train_generator.class_indices.keys())
    return class_names[predicted_class], confidence

print("\nModel training completed!")
print("Expected accuracy: >90%")
print("Use predict_tomato_stage(image_path) to make predictions on new images")

Found 150 images belonging to 5 classes.
10/10 ━━━━━━━━━━━━━━━━━━━━ 51s 4s/step - accuracy: 0.9065 - loss: 0.5381
Final Validation Accuracy: 85.33%

Applying Test Time Augmentation...
Found 150 images belonging to 5 classes.
Found 150 images belonging to 5 classes.
Found 150 images belonging to 5 classes.
Found 150 images belonging to 5 classes.
Found 150 images belonging to 5 classes.
Found 150 images belonging to 5 classes.
Found 150 images belonging to 5 classes.
Found 150 images belonging to 5 classes.
Found 150 images belonging to 5 classes.
